<a href="https://colab.research.google.com/github/FishyFoshy/COMP3608-Project/blob/main/Dataset_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
import numpy as np

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# Load datasets
df2 = pd.read_csv('Dataset 2.csv')

# Drops all unnecessary columns
df2 = df2.drop(columns=['age']) 

# Drops all with null values
df2.dropna(inplace=True)

# Separates non numerical columns from numerical ones
non_numerical_columns = ['status', 'price', 'location', 'builder']
numerical_features = [col for col in df2.columns if col not in non_numerical_columns]

# Scale numerical features
scaler = StandardScaler()
df2[numerical_features] = scaler.fit_transform(df2[numerical_features])

# One-Hot Encodes all categorical data
categorical_cols_to_encode = df2.select_dtypes(include='object').columns
df2 = pd.get_dummies(df2, columns=categorical_cols_to_encode, drop_first=True)

# Identifies target column from the features
target_column = 'price'
features = [col for col in df2.columns if col != target_column]

x = df2[features].copy()
y = df2[target_column].copy()

# Splits the data 80/20 for training and testing the model respectfully
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("=" * 60)
print("DATASET 2: Dataset 2.csv")
print("=" * 60)
print("\nFirst 10 rows:")
display(df2.head(10))
print("\nData types:")
print(df2.dtypes)
print("\nMissing values:")
print(df2.isnull().sum())
print("\nTarget variable (price) statistics:")
print(df2['price'].describe())

# Price is in Lahk which is 100,000 Rupees (₹)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("\n -------Linear Regression Model-----")


#Initializing Linear Regression Model and Running
model = LinearRegression()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)


#Fetching the Performance Metrics For The Model
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#Graph Showing predicted vs Actual With Errors and Coefficents

#Printing Performance Metrics for the model
print(f"Mean Squared: {mse:.2f}\n")
print(f"Mean Absolute Error: {mae:.2f}\n")
print(f"R^2 Score: {r2:.2f}\n")

#Constructing A Scatter Plot of Real Vs Expected Value to Vizualize the Perfomrance of the Model on The Test Set
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color='blue')
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2)
plt.xlabel('Actual Prices (Lakh ₹)', fontsize=12)
plt.ylabel('Predicted Prices (Lakh ₹)', fontsize=12)
plt.title('Actual vs. Predicted Property Prices', fontsize=14)
plt.grid(True, linestyle=':', alpha=0.7)
plt.show()

In [ ]:
#Fetching Feature Names and Coefficents
feature_names = x_train.columns
coefficients = model.coef_

#Wrapping them together in a Pandas DataFrame for Easy Comparision and Viewing
feature_importance = pd.DataFrame({
 'Feature': feature_names,
 'Coefficient': coefficients
})

#Using the Absolute Function in the NumPy Library before Sorting the Coefficents in Decending Order
feature_importance['Abs_Coefficient'] = np.abs(feature_importance['Coefficient'])
top_10_features = feature_importance.sort_values(by='Abs_Coefficient', ascending=False).head(10)

#Plotting Graph
plt.figure(figsize=(10, 6))
sns.barplot(
 data=top_10_features, 
 x='Coefficient', 
 y='Feature', 
)
plt.title('Top 10 Features Driving Property Prices', fontsize=15)
plt.xlabel('Coefficient Value (Impact on Price in Lakh ₹)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(True, axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
display(top_10_features[['Feature', 'Coefficient']])


The Linear Regression Model is able to predict the prices of cheap homes/standard very well which accounts for it's very high .85 R^2 score. However as the price of the properties begin to increase the model begins to break down. The most important features seem to be the realestate companies themselves and location. As these were the features which made it into the top 10. This may indicate that for the specific data set "builder_Shree sakthivel realestate" and "builder_Vinay Asrani" may have made up a majority of the middle-high real estate or they only had middle-high real estate for this sample. The model then began to become dependent on these 2 features to make predictions on the higher cost housing. Which may explain the low accruacy once the price of the houses crossed 200 Lakh. This may indicate that a dataset with more higher end housing from different real estate companies maybe required. When the Datasets are joined we should expect the R^2 value to reduce but the accruacy at the top of the price bracket should improve as the model is able to find more reliable predictors. This hyptothesis is further backed up by a MSE that greatly exceeds the MAE. The MAE is fairy small and managable since it doesn't penalize large deviations to the same extent as the MSE which is order of magnitudes larger using the root MSE you still get a value that is over double the MAE further showing the ineffectivness of the model at the the higher end of the dataset.